In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import unidecode
from collections import Counter

In [6]:
df = pd.read_csv("../../data/data_cleaned/patients_adresse_id.csv", dtype={'pseudo_provisoire':str,
                                                                     'requete': str,
                                                                     'adresse':str,
                                                                     'codepost':str,
                                                                     'nom_commune_postal':str},sep=";")

### Occurences des différents biais dans nos données : 

In [7]:
bruit_int = ["APPT","BAT","CENTRE","CHEZ","HOPITAL","HOTEL","MAISON","MME","MR", "QUARTIER","RES","RESIDENCE","RETRAITE"]


df_biais = df[df['bruit']!=""][['pseudo_provisoire','bruit']].copy()
df_biais['bruit'] = df_biais['bruit'].str.replace(' ','')

df_biais[['bruit_1','bruit_2','bruit_3','bruit_4']] = df_biais['bruit'].str.split(',',expand =True)

all_bruit = pd.concat([df_biais['bruit_1'], df_biais['bruit_2'], df_biais['bruit_3'], df_biais['bruit_4']])

occurrences_bruit = pd.DataFrame(all_bruit.value_counts())
occurrences_bruit = occurrences_bruit[occurrences_bruit.index.isin(bruit_int)].reset_index()
occurrences_bruit.columns=['bruit','count']

In [ ]:
import plotly.express as px

fig = px.histogram(occurrences_bruit, x='bruit', y = 'count',title ="Nombre d'occurence de chaque biais dans nos adresse")
fig.update_yaxes(title="Occurence")
fig.update_xaxes(title="Bais identifiés")
fig.show()
# fig.write_image("images/histo_biais_occ.jpeg", scale =2)


### Freq des éléménts dans nos données : 

In [ ]:
mots = df["requete"].str.split(expand=True).stack()
mot_sans_accents = [ unidecode.unidecode(mot) for mot in mots]

mots_up = [mot.upper() for mot in mot_sans_accents]

count = Counter(mots_up) 
mots_freq = count.most_common()
co = pd.DataFrame(mots_freq, columns = ["token", "count"])

res = co[co["count"]>500]

In [ ]:
import plotly.graph_objects as go
fig = go.Figure(data=[
                     go.Table(
                        header=dict(values=list(res.columns),align='center'),
                        cells=dict(values=res.values.transpose(),
                                   fill_color = [["white","lightgrey"]*res.shape[0]],
                                   align='center'
                                  )
                            )
                       ])
fig.update_layout(
    autosize=False,
    margin = {'l':0,'r':0,'t':0,'b':0},
    height = 600
)

# fig.show()
# fig.write_image('./images/token_occurence.png', scale =1)

### % apparition des différentes classes de l'adresse

In [ ]:
street_w_num = len(df[df['numeros'] !=""])
prc_street_w_num = (street_w_num/len(df)) *100

street_w_type = len(df[df['voirie'] !=""])
prc_street_w_type = (street_w_num/len(df)) *100


street_w_name = len(df[df['elem_adresse'] !=""])
prc_street_w_name = (street_w_name/len(df)) *100


street_w_cpville = len(df[df['codepost'] !=""])
prc_street_w_cpville = (street_w_name/len(df)) *100

street_w_city = len(df[df['nom_commune_postal']!=""])
prc_street_w_city = (street_w_city/len(df)) *100


street_w_bruit = len(df[df['bruit']!=""])
prc_street_w_bruit = (street_w_bruit/len(df)) *100



### Position des biais dans nos adresses 

In [10]:
bruit_int = ["APPT","BAT","CENTRE","CHEZ","HOPITAL","HOTEL","MAISON","MME","MR", "QUARTIER","RES","RESIDENCE","RETRAITE"]

df_bruit_av_ap = pd.DataFrame(np.zeros((len(bruit_int),3)),index=bruit_int,columns=['AV','AP','SNV'])

for i in df[~pd.isna(df['bruit'])].index : 
    pos_bruit_list = df.loc[i,'pos_bruit'].split(',')
    adresse_part = df.loc[i,'adresse'].split(' ')

    if len(pos_bruit_list) ==1 : 
        bruit = df.loc[i,'bruit'].replace(' ','')
        if bruit in bruit_int : 
            if pd.isna(df.loc[i,'bruit_AV_AP']):
                if pd.isna(df.loc[i,'pos_numeros']) and pd.isna(df.loc[i,'pos_voirie']):
                    df_bruit_av_ap.loc[bruit.replace(' ',''),'SNV']+=1
                if pd.isna(df.loc[i,'pos_numeros']) and pd.isna(df.loc[i,'pos_voirie']) is False:
                    pos_voirie = int(df.loc[i,'pos_voirie'])
                    pos_bruit = adresse_part.index(bruit) +1
                    if pos_voirie < pos_bruit:
                        df_bruit_av_ap.loc[bruit.replace(' ',''),'AP'] +=1
                    if pos_voirie > pos_bruit:
                        df_bruit_av_ap.loc[bruit.replace(' ',''),'AV']+=1
                if pd.isna(df.loc[i,'pos_numeros']) is False:
                    pos_numeros = int(df.loc[i,'pos_numeros'])
                    pos_bruit = adresse_part.index(bruit) +1
                    if pos_numeros < pos_bruit:
                        df_bruit_av_ap.loc[bruit.replace(' ',''),'AP']+=1
                    if pos_numeros > pos_bruit:
                        df_bruit_av_ap.loc[bruit.replace(' ',''),'AV']+=1
            else : 
                bruit_AV_AP = df.loc[i,'bruit_AV_AP'].replace(' ','')
                df_bruit_av_ap.loc[bruit,bruit_AV_AP] +=1                                             

    if len(pos_bruit_list) >1 :
        bruit_list = df.loc[i,'bruit'].split(',')
        for bruit in bruit_list: 
            if bruit in bruit_int and bruit in adresse_part:
                if pd.isna(df.loc[i,'pos_numeros']) and pd.isna(df.loc[i,'pos_voirie']):
                    df_bruit_av_ap.loc[bruit.replace(' ',''),'SNV']
                if pd.isna(df.loc[i,'pos_numeros']) and pd.isna(df.loc[i,'pos_voirie']) is False:
                    pos_voirie = int(df.loc[i,'pos_voirie'])
                    pos_bruit = adresse_part.index(bruit) +1
                    if pos_voirie < pos_bruit:
                        df_bruit_av_ap.loc[bruit.replace(' ',''),'AP']+=1
                    if pos_voirie > pos_bruit:
                        df_bruit_av_ap.loc[bruit.replace(' ',''),'AV']+=1
                if pd.isna(df.loc[i,'pos_numeros']) is False:
                    pos_numeros = int(df.loc[i,'pos_numeros'])
                    pos_bruit = adresse_part.index(bruit) +1
                    if pos_numeros < pos_bruit:
                        df_bruit_av_ap.loc[bruit.replace(' ',''),'AP']+=1
                    if pos_numeros > pos_bruit:
                        df_bruit_av_ap.loc[bruit.replace(' ',''),'AV']+=1

df_bruit_av_ap[['AV','AP','SNV']] = df_bruit_av_ap[['AV','AP','SNV']].astype(int)#= pd.DataFrame(np.zeros((len(bruit_int),3)),index=bruit_int,columns=['AV','AP','AV_AP','SNV'])
df_bruit_av_ap
        # print(df.loc[i,:])

,AV,AP,SNV
APPT,49,679,0
BAT,71,525,3
CENTRE,9,6,0
CHEZ,639,143,7
HOPITAL,8,1,1
HOTEL,2,4,1
MAISON,50,45,6
MME,23,8,0
MR,9,5,0
QUARTIER,4,8,0


In [1]:
df_bruit_av_ap

NameError: name 'df_bruit_av_ap' is not defined

### recherche de la position du code postal et de la ville dans l'adresse 


In [ ]:
# df = df.head()

df["pos_cp"]=np.nan
df["pos_ville"]=np.nan

for i in df.index: 

    if str(df.loc[i,"codepost"])[-2:] ==".0":
        df.loc[i,"codepost"] = str(df.loc[i,"codepost"])[:-2]
    
    cp = str(df.loc[i,"codepost"])#[:-2]
    ville = df.loc[i,'nom_commune_postal']

    adresse_part = str(df.at[i,"requete"]).split(" ")
    adresse_part = ' '.join(adresse_part).split()
    long_ad = len(adresse_part)
    
    for j, adresse_part in enumerate(adresse_part):
        if adresse_part==str(cp):
            df.loc[i,"pos_prc_cp"]= ((j+1)/long_ad)*100
        if adresse_part ==ville:
            df.loc[i,"pos_prc_ville"] =((j+1)/long_ad)*100
        





In [ ]:
## selon le nombre de bruit max : ici 6
df[['pos_bruit_1', 'pos_bruit_2','pos_bruit_3','pos_bruit_4','pos_bruit_5','pos_bruit_6']] = df['pos_prc_bruit'].str.split(',', expand=True)

df[df["pos_bruit_1"]==""] = np.nan

df["pos_bruit_1"] = df["pos_bruit_1"].astype(float)

df[df["pos_bruit_2"]==""] = np.nan

df["pos_bruit_2"] = df["pos_bruit_2"].astype(float)

df[df["pos_bruit_3"]==""] = np.nan

df["pos_bruit_3"] = df["pos_bruit_3"].astype(float)

df[df["pos_bruit_4"]==""] = np.nan

df["pos_bruit_4"] = df["pos_bruit_4"].astype(float)

df[df["pos_bruit_5"]==""] = np.nan

df["pos_bruit_5"] = df["pos_bruit_5"].astype(float)

df[df["pos_bruit_6"]==""] = np.nan

df["pos_bruit_6"] = df["pos_bruit_6"].astype(float)



In [ ]:
for i in df.index : 
    if pd.isna(df.loc[i,"pos_prc_voirie"]) is False : 
        if len(df.loc[i,"pos_prc_voirie"].split(','))>1 :
            pos_voirie = df.loc[i,"pos_prc_voirie"].split(',')
            df.loc[i,"pos_prc_voirie"] = pos_voirie[0]


## Normalisation : 

In [ ]:
df['pos_prc_voirie'] = pd.to_numeric(df['pos_prc_voirie'])
df['pos_prc_numeros'] = pd.to_numeric(df['pos_prc_numeros'])
df['pos_prc_ville'] = pd.to_numeric(df['pos_prc_ville'])

df['pos_prc_bruit'] = df[['pos_bruit_1','pos_bruit_2','pos_bruit_3','pos_bruit_4']].mean(axis=1)


mean_voirie = df['pos_prc_voirie'].mean(skipna=True)
std_voirie = df['pos_prc_voirie'].std(skipna=True)
df['voirie_norm'] = df['pos_prc_voirie'] - mean_voirie / std_voirie

mean_num = df['pos_prc_numeros'].mean(skipna=True)
std_num = df['pos_prc_numeros'].std(skipna=True)
df['num_norm'] = df['pos_prc_numeros'] - mean_num / std_num

mean_ville = df['pos_prc_ville'].mean(skipna=True)
std_ville = df['pos_prc_ville'].std(skipna=True)
df['ville_norm'] = df['pos_prc_ville'] - mean_ville / std_ville


mean_bruit = df['pos_prc_bruit'].mean(skipna=True)
std_bruit = df['pos_prc_bruit'].std(skipna=True)
df['bruit_norm'] = df['pos_prc_bruit'] - mean_bruit / std_bruit

mean_cp = df['pos_prc_cp'].mean(skipna=True)
std_cp = df['pos_prc_cp'].std(skipna=True)
df['cp_norm'] = df['pos_prc_cp'] - mean_cp / std_cp




In [ ]:
#                'ville_norm':'City'

df = df.rename({'voirie_norm':"Street type",
                "num_norm": "Numbers",
                "bruit_norm": "Add. info",
                'cp_norm': 'Zipcode'},axis=1)
# Fusionner les données de position pour la visualisation
positions = pd.melt(df, id_vars=['adresse'], value_vars=['Numbers','Street type','Zipcode','Add. info'], #'City'
                    var_name='Type', value_name='Position_relative')


# Assurez-vous que la 'Position relative' est de type numérique
positions['Position_relative'] = positions['Position_relative'].astype(float)

In [ ]:
sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

# Initialize the FacetGrid object
pal = sns.cubehelix_palette(7, rot=-.25, light=.4)
g = sns.FacetGrid(positions, row="Type", hue="Type", aspect=5, height=1, palette=pal)

# Draw the densities in a few steps
g.map(sns.kdeplot, "Position_relative",
      bw_adjust=.5, clip_on=False,
      fill=True, alpha=1, linewidth=1.5)
# g.map(sns.kdeplot, "Position_relative", clip_on=False, color="w", bw_adjust=.5) # lw=2,

# passing color=None to refline() uses the hue mapping
g.refline(y=0, linewidth=2, linestyle="-",color=None, clip_on=False)  #


# Define and use a simple function to label the plot in axes coordinates
def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)


g.map(label, "Position_relative")

# Set the subplots to overlap
g.figure.subplots_adjust(hspace=-.25)

# Remove axes details that don't play well with overlap
g.set_titles("")
g.set(yticks=[], ylabel="", xlabel="Relative Position (%)")
g.despine(bottom=True, left=True)

In [ ]:



# Graphique de densité corrigé
sns.kdeplot(data=positions, x='Position_relative', hue='Type', fill=True, common_norm=False, alpha=0.5)
plt.title('Density of relative position of adresses elements')#Densité de la position relative des éléments de l\'adresse')
plt.xlabel('Relative position (%)')
plt.ylabel('Density')
plt.show()


In [ ]:
## 
import geopandas as gpd

In [ ]:
ad_1 = {'id':"0",'adresse': "23 rue de Rivoli",'x':"2.3569522395967533",'y':"48.85613014384232"}
 
ad_2 = {'id':"1",'adresse': "39 rue de Rivoli",'x':"2.3494418089223235",'y':"48.858184618699674"}
 
ad_3 = {'id':"2",'adresse': "134 rue de Rivoli",'x':"2.34372136049349",'y':"48.8599756791133"}


test = pd.DataFrame.from_dict([ad_1, ad_2,ad_3])

gdf = gpd.GeoDataFrame(
    test, geometry=gpd.points_from_xy(test.x, test.y), crs="EPSG:4326"
)
gdf.to_file("H:/canc_air/data/zones_geographiques/rivoli/rivolis.shp")